# Talker vs. Doer: from a frozen model to a minimal agent loop

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 1 §1.1–1.2, §1.5 · type: concept-lab*

**The promise:** run the *same task* two ways — one bare model call vs. a ~15-line reason→act→observe loop — and point, in a printed trace, at exactly where the model supplies *judgment* and where *your* code supplies the loop, tools, and guardrails.

This is one of the first runnable notebooks in the whole repo. It runs **fully offline and free** — `MOCK=1` (the default) returns canned, realistic model turns, so you need no API key and nothing is billed.

## 🧠 Why this matters

A language model on its own is a *function*: text in, text out. It is frozen — it cannot look anything up, run code, or check whether its answer was right (§1.1). An **agent** wraps that frozen function in a loop: give it a goal and some tools, and let it decide, step by step, what to do next — call a tool, read the result, decide it's done, then answer.

The reframing the chapter builds everything on: the model is a **reasoning engine**, and your system is the **body** it drives — the body *senses* (gathers context), *acts* (runs tools), and *remembers* (persists state). A chatbot answers; an agent *accomplishes*. This notebook makes that one sentence runnable, so you *feel* the difference before any framework or production concern arrives. See §1.1–1.2.

## Objectives & prereqs

**By the end you can:**
- Run a bare model call and watch it *guess or refuse* on a question it cannot look up.
- Wrap that same model in a minimal **reason→act→observe** loop with one tiny tool, and read the trace where it calls the tool, observes the result, then answers.
- Score a run against the chapter's **four agentic properties** — autonomy, tool use, goals, feedback loops — and find each one in the trace.
- Slide the autonomy *knob* (`max_steps`) and explain why agentic is a **spectrum, not a label**.

**Prereqs:** Chapter 1 read. No prior notebook required — this is a safe first run (offline, no key, no services).

**Packages:** standard library only. The optional live path imports `anthropic` (in `requirements.txt`) but is never needed in the default `MOCK=1` mode.

In [ ]:
# --- Setup -------------------------------------------------------------------
import os
import random

from dotenv import load_dotenv

load_dotenv()  # reads a local .env if present; never hardcode keys

# MOCK=1 (default): canned, realistic model turns -> free, offline, deterministic.
# MOCK=0: hit the live Anthropic API (~2 short calls). Documented, never required.
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

random.seed(1)  # determinism for anything stochastic in this notebook

MODEL = "claude-opus-4-8"  # the book's default model

if MOCK:
    print("MOCK mode: scripted model turns. No API key needed, nothing billed.")
else:
    if not os.getenv("ANTHROPIC_API_KEY"):
        raise RuntimeError(
            "COMPANION_MOCK=0 but ANTHROPIC_API_KEY is not set. "
            "Set it in your environment/.env, or use COMPANION_MOCK=1 (offline)."
        )
    import anthropic

    client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from the environment
    print(f"LIVE mode: calling {MODEL}. About 2 short calls will be billed.")

## The task (one goal, two ways)

We'll give the *same* goal to a talker and a doer:

> *"What is the current internal status of service ORION-7, and is it safe to deploy?"*

That status lives in an **internal** system the model has never seen — it isn't in any training data and can't be guessed. There is exactly one honest way to know it: **look it up**. That's what separates a talker from a doer.

Our tiny "internal system" is just an in-memory dict — 2–3 facts, no external services. (A real tool would hit a database or an API; the *shape* is identical.)

In [ ]:
# The "internal system" the model cannot know from training: a tiny fact store.
SERVICE_STATUS = {
    "ORION-7": {"status": "degraded", "error_rate": 0.12, "deploy_freeze": True},
    "ORION-3": {"status": "healthy", "error_rate": 0.001, "deploy_freeze": False},
    "VEGA-1": {"status": "healthy", "error_rate": 0.004, "deploy_freeze": False},
}

GOAL = "What is the current internal status of service ORION-7, and is it safe to deploy?"
print("GOAL:", GOAL)

## 1 · The talker: a single, frozen model call

First the talker — one model call, no tools, no loop. It answers from whatever is baked into its weights and then stops. Below, `MOCK=1` returns a canned, realistic answer for exactly this prompt; `MOCK=0` issues one real call. Either way, the model has **no access** to `SERVICE_STATUS` — it isn't part of the prompt and the model has no way to reach it.

In [ ]:
def talker(goal: str) -> str:
    """One frozen model call: text in, text out. No tools, no loop.

    MOCK=1 returns a canned, realistic 'I can't actually look that up' answer.
    MOCK=0 issues a single real Anthropic call. The point of the cell is the
    *shape* of the failure, which is identical either way.
    """
    if MOCK:
        return (
            "I don't have access to your internal systems, so I can't look up the live "
            "status of ORION-7 or whether a deploy freeze is in effect. In general, you "
            "should avoid deploying to a degraded service — but I can't confirm its "
            "current state from here."
        )
    resp = client.messages.create(
        model=MODEL,
        max_tokens=300,
        messages=[{"role": "user", "content": goal}],
    )
    return resp.content[0].text


print(talker(GOAL))

## 🔮 Predict: can the talker answer?

Before you re-read the output: the question needs a **live lookup** into a system the model has never seen. **Predict** which of these the talker does —

- (a) confidently states ORION-7's real status (`degraded`, freeze on), or
- (b) refuses / hedges because it has no access, or
- (c) *guesses* a plausible-but-unverifiable status.

Write your guess, then read the answer above again. What it **cannot** do is the whole motivation for an agent: an honest model hits a wall (b/c); it has no mechanism to *act* and find out (a).

**What you just saw.** The talker is well-behaved — it admits it can't reach your systems — but it is *stuck*. There is no path from "I don't know" to "let me find out," because a bare model call has no tools and no second step. Everything it knows ended the moment it started generating. To get the real answer we have to give it a body.

## 2 · One tiny tool, then the doer

A **tool** is just a Python function the model is allowed to call. Here's one: a lookup over our in-memory `SERVICE_STATUS`. Notice it's plain code — *your* code — and it's where the agent touches the real world.

In [ ]:
def lookup_status(service: str) -> str:
    """The agent's one tool: read live status from the internal fact store."""
    record = SERVICE_STATUS.get(service.strip().upper())
    if record is None:
        return f"No such service: {service!r}. Known: {list(SERVICE_STATUS)}"
    return (
        f"status={record['status']}, error_rate={record['error_rate']:.1%}, "
        f"deploy_freeze={record['deploy_freeze']}"
    )


# Smoke-test the tool directly (no model involved yet).
print("ORION-7 ->", lookup_status("ORION-7"))
print("VEGA-1  ->", lookup_status("VEGA-1"))

### The reason→act→observe loop

Now the doer. It is deliberately tiny — about fifteen lines, **no framework** — so the mechanism stays visible. Each pass: ask the model what to do (**reason**); if it asks for a tool, run it (**act**) and feed the result back (**observe**); repeat until the model says it's done, or we hit `max_steps`.

The `_model_turn` below is the *reasoning engine*. In `MOCK=1` it's a scripted stand-in that drives a realistic 2-step run: first it asks to look up ORION-7, then — having observed the result — it answers. The **loop, the tool wiring, and the step bound are all your code**; the model only decides *what to do next*.

In [ ]:
TOOLS = {"lookup_status": lookup_status}


def _model_turn(goal: str, observations: list[str]) -> dict:
    """The reasoning engine: given the goal and what we've observed so far,
    decide the next step. Returns either an action or a final answer.

    MOCK=1: a scripted stand-in (no key). MOCK=0: a real model call whose text
    we parse into the same {action|answer} shape. The decision is the model's;
    the loop around it is ours.
    """
    if MOCK:
        if not observations:
            # Reason: "I need the live status -> call the tool."
            return {"type": "action", "tool": "lookup_status", "arg": "ORION-7",
                    "thought": "I can't know this from memory; I'll look it up."}
        # Observe the tool result, then answer.
        seen = observations[-1]
        verdict = "NOT safe to deploy" if "deploy_freeze=True" in seen else "safe to deploy"
        return {"type": "answer",
                "text": f"ORION-7 is currently {seen}. It is {verdict} — a deploy "
                        f"freeze is in effect and the error rate is elevated."}
    # --- live path: ask the model to emit a one-line directive we can parse ---
    sys = (
        "You can call one tool: lookup_status(service). To use it, reply EXACTLY "
        "'ACT lookup_status <service>'. When you can answer, reply 'ANSWER <text>'."
    )
    convo = goal + "".join(f"\nOBSERVED: {o}" for o in observations)
    resp = client.messages.create(
        model=MODEL, max_tokens=300, system=sys,
        messages=[{"role": "user", "content": convo}],
    )
    line = resp.content[0].text.strip()
    if line.startswith("ACT "):
        _, tool, arg = line.split(" ", 2)
        return {"type": "action", "tool": tool, "arg": arg, "thought": line}
    return {"type": "answer", "text": line.removeprefix("ANSWER ").strip()}


def doer(goal: str, max_steps: int = 3) -> tuple[str, list[str]]:
    """A minimal reason->act->observe agent. Returns (answer, trace)."""
    observations: list[str] = []
    trace: list[str] = []
    for step in range(1, max_steps + 1):           # the step bound is a guardrail
        decision = _model_turn(goal, observations)  # REASON (model's judgment)
        if decision["type"] == "answer":
            trace.append(f"step {step}: ANSWER")
            return decision["text"], trace
        tool, arg = decision["tool"], decision["arg"]
        trace.append(f"step {step}: REASON -> {decision['thought']}")
        result = TOOLS[tool](arg)                   # ACT (your code touches the world)
        observations.append(result)                 # OBSERVE (feed the result back)
        trace.append(f"step {step}: ACT {tool}({arg!r}) -> OBSERVE {result!r}")
    return "Stopped: step limit reached without a final answer.", trace

## 🔮 Predict: what will the doer's trace look like?

Same goal, same model — but now there's a loop and a tool. **Predict before running:**

1. How many steps until it answers?
2. Does the final answer correctly conclude it's **not** safe to deploy?
3. Which line in the trace is the model's *judgment*, and which is *your code* acting?

Then run it and read the trace line by line.

In [ ]:
answer, trace = doer(GOAL)

print("--- TRACE ---")
for line in trace:
    print(line)
print("\n--- FINAL ANSWER ---")
print(answer)

**What you just saw.** Same frozen model, completely different outcome. On step 1 it *reasoned* that it needed live data and emitted an action; **your loop** ran the tool and fed the result back; on step 2 it *observed* that result and produced a grounded, correct answer. The model supplied the **judgment** ("I should look this up"; "a freeze means don't deploy"). Everything else — actually calling `lookup_status`, capturing the observation, looping, stopping — was **your code**. That is the whole thesis of §1.1: *agent = model + loop + tools + guardrails*.

## 3 · Score the run against the four agentic properties

§1.2 pins down what "agentic" means with four properties. Let's find each one in the trace we just produced rather than take them on faith. (We detect them structurally from the run — no model needed.)

In [ ]:
def score_properties(trace: list[str]) -> dict:
    """Locate the four agentic properties (§1.2) as evidence in a run's trace."""
    acted = [t for t in trace if "ACT " in t]
    observed = [t for t in trace if "OBSERVE" in t]
    return {
        # Autonomy: the model CHOSE to call a tool; the step count wasn't hard-coded.
        "autonomy":      bool(acted),
        # Tool use: it affected the world beyond generating text.
        "tool_use":      bool(acted),
        # Goals: it was pointed at an objective, not a single chat turn.
        "goals":         True,
        # Feedback loops: it OBSERVED a result and adapted its next step.
        "feedback_loop": bool(observed),
    }


scores = score_properties(trace)
for prop, present in scores.items():
    print(f"  {'✅' if present else '— '} {prop}")

print("\nThe talker would score True only on 'goals' — no autonomy, no tools, no loop.")

**What you just saw.** The doer lights up all four properties; the talker would light up only *goals*. Those four — **autonomy, tool use, goals, feedback loops** — are exactly the chapter's definition, and you can now point at each one in a trace you produced.

## 4 · The autonomy knob: agentic is a spectrum, not a label

§1.2's key idea: *agentic is a spectrum, not a label.* The same code is *more or less* agentic depending on how much freedom you grant it. The crudest knob is `max_steps` — how many reason→act cycles it's allowed. Cap it at **1** and the agent can reason once but never gets to observe and answer; give it **3** and it completes. **Predict** what each cap prints, then run.

In [ ]:
for cap in (1, 3):
    ans, tr = doer(GOAL, max_steps=cap)
    print(f"max_steps={cap}: {len(tr)} trace lines -> {ans[:60]}{'...' if len(ans) > 60 else ''}")
    print()

**What you just saw.** With `max_steps=1` the agent reasons and acts but is cut off before it can observe-and-answer — barely agentic. With `max_steps=3` it completes the loop. Autonomy is a *dial you set*, not a property the system either has or lacks. Your engineering job (§1.2) is to pick the right point on that spectrum — **and not a drop more autonomy than you can evaluate and control.**

## ⚠️ Pitfall: mistaking this *demo* for a *system*

This toy is a great teaching artifact and a **terrible production agent** — and the gap between them is the whole rest of the book (§1.5). Look at everything this happy-path demo silently skips:

- **No input validation.** The model could ask to look up a service that doesn't exist, or pass junk; we'd hand it straight to the tool.
- **No error handling.** If `lookup_status` raised, the loop would crash — not recover.
- **No cost or latency control.** Real model calls cost money and time; there's no budget here.
- **No evaluation.** We *eyeballed* one correct answer. How do we know it's right across a hundred services?
- **No observability, auth, or guardrails on risky actions.** A real "is it safe to deploy?" agent would need a human gate before it could *act* on that answer.

Mistaking a dazzling happy-path demo for a dependable system is the most common early mistake. Parts II–VI are precisely those hundred unglamorous decisions — which is where the value, and the career, actually live.

## 🎯 Senior lens

Read the loop again with one phrase in mind: *not a drop more autonomy than you can evaluate and control.* Every degree of freedom you just handed the model — *whether* to call a tool, *which* tool, *what* argument, *how many* steps — is a degree of freedom you must be able to **observe, evaluate, and bound**. That's why a senior reaches for the `max_steps` guard before the agent is clever, not after it loops twelve times in production; why the tool result is captured into a trace (you can't debug or evaluate what you didn't record); and why "is it safe to deploy?" is the kind of *action* that earns a human-approval gate, not blind execution. The skill this book builds isn't "making the agent more autonomous" — it's *systems engineering applied to a non-deterministic component*: matching autonomy to the evaluation and control you can actually provide.

## Recap

- A bare model is a **frozen function** (text in, text out): it can talk, but it can't *look up*, *act*, or *check* — so it gets stuck on anything needing live data.
- An **agent = model + loop + tools + guardrails.** The model supplies *judgment*; your code supplies the **reason→act→observe** loop, the tools, and the bounds.
- The **four agentic properties** — autonomy, tool use, goals, feedback loops — are visible as concrete lines in a run's trace; the talker has only *goals*.
- **Agentic is a spectrum, not a label**: `max_steps` is the crudest autonomy knob, and choosing where to sit on that spectrum is the engineer's job.
- A happy-path **demo is not a system** — validation, errors, cost, eval, and guardrails (the rest of the book) are what make it one.

## Exercises

Use the empty cells below. Each one *changes* something — predict the result before you run. (Solutions land in `solutions/` in Phase 2.)

1. **Add a second tool.** Give the agent a `safe_to_deploy(service) -> bool` tool that encodes the rule (no deploy if `deploy_freeze` is on *or* `error_rate > 0.05`). Have the scripted model call *both* tools before answering. Predict: how many steps now, and which agentic property does adding a second tool most strengthen?
2. **Where would this break in production?** Pick three items from the ⚠️ pitfall list and, for each, write the one line of the loop or tool you'd have to change first. Predict which failure would bite *soonest* under real traffic.
3. **Move along the spectrum.** Ask the agent about a service that *isn't* in the fact store (e.g. `"NEBULA-9"`) using the live-style directive shape. Predict what the tool returns and whether the agent recovers or loops to `max_steps`. What guardrail from the senior-lens cell would you add?
4. **(Live, optional)** Set `COMPANION_MOCK=0` with `ANTHROPIC_API_KEY` set and run both `talker` and `doer`. Does the real model refuse the bare question? Does it correctly emit an `ACT` directive in the loop? Note where the live trace differs from the scripted one.

In [ ]:
# Exercise 1 — your code here


In [ ]:
# Exercise 2 — your notes/code here


In [ ]:
# Exercise 3 — your code here


## Next

You just built the *seed* of every agent in this repo. Here's where it grows.

- ▶️ **Next notebook:** [`../03-mental-model/03-01-four-planes-traced.ipynb`](../03-mental-model/03-01-four-planes-traced.ipynb) — the other Part I orientation lab: trace one request through the four planes (model · orchestration · data · infrastructure).
- 🔧 **The real version of this toy loop:** [`../../blueprints/agent-loop/`](../../blueprints/agent-loop/) — the production agent loop (typed tools, retries, telemetry), built for real in **Ch 12**. This notebook is its conceptual seed.
- 🎓 **Capstone:** [`../../capstone-project/`](../../capstone-project/) — you'll build *your own* `agents/raw/` loop starting in Ch 12; the frame is **build yours first**, with the capstone as an answer key, never a copy.
- 📖 **Book:** revisit §1.1–1.2 (models → agents; the four properties) and §1.5 (demo vs. system).